# FILE 2: preprocessing_and_feature_engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Load Dataset
df = pd.read_csv("/content/house_price_regression_dataset.csv")
df.head()


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06


**Split Train / Test**

In [ ]:
# X = features
X = df.drop("House_Price", axis=1)

# y = target
y = df["House_Price"]

# Split: 80% Train + 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (800, 7)
X_test shape: (200, 7)
y_train shape: (800,)
y_test shape: (200,)


**Split Train into Train & Validation**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)

print("Train/Validation Split Done!")
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)


Train/Validation Split Done!
X_train shape: (640, 7)
X_val shape: (160, 7)
y_train shape: (640,)
y_val shape: (160,)


**Feature Engineering Function**

In [ ]:
import numpy as np
import pandas as pd

def add_features(X):
    # We use .copy() to ensure that the original DataFrame X is not changed.
    X = X.copy()

    # =========================
    # 1) Basic Structural Features
    # =========================
    # Calculate the current age of the house. (Assuming the current year is 2025)
    X["House_Age"] = 2025 - X["Year_Built"]

    # Total rooms (number of bedrooms + number of bathrooms)
    X["Total_Rooms"] = X["Num_Bedrooms"] + X["Num_Bathrooms"]

    # The ratio of the number of bedrooms to the number of bathrooms.
    # We use np.where to avoid division by zero (in the case of Num_Bathrooms = 0), and in this case, we use NaN.
    X["Bed_Bath_Ratio"] = np.where(
        X["Num_Bathrooms"] > 0,
        X["Num_Bedrooms"] / X["Num_Bathrooms"],
        np.nan
    )

    # Lot size conversion: Convert Lot_Size (in acres) to square meters (sqm).
    X["Lot_Size_sqm"] = X["Lot_Size"] * 4046.8564224

    # Rooms Density: (Total Rooms / Square Footage).
    # Measures how 'crowded' the rooms are within the area. np.where avoids division by zero.
    X["Rooms_Density"] = np.where(
        X["Square_Footage"] > 0,
        X["Total_Rooms"] / X["Square_Footage"],
        np.nan
    )

    # Ratio of garage size (car capacity) to total rooms.
    X["Garage_per_Room"] = np.where(
        X["Total_Rooms"] > 0,
        X["Garage_Size"] / X["Total_Rooms"],
        np.nan
    )

    # Neighborhood Size Index: (Neighborhood Quality * Square Footage).
    # Combines the size of the house with the surrounding neighborhood quality.
    X["Neighborhood_Size"] = X["Neighborhood_Quality"] * X["Square_Footage"]

    # Structural Value Index: (Square Footage * Quality) divided by Total Rooms.
    # Measures the efficiency of space utilization given the neighborhood quality.
    X["Structural_Value_Index"] = np.where(
        X["Total_Rooms"] > 0,
        (X["Square_Footage"] * X["Neighborhood_Quality"]) / X["Total_Rooms"],
        np.nan
    )

    # Luxury Index: (Garage Size + Number of Bathrooms) * Neighborhood Quality.
    # Focuses on high-value, comfort features of the property and area.
    X["Luxury_Index"] = (X["Garage_Size"] + X["Num_Bathrooms"]) * X["Neighborhood_Quality"]

    # Price Size Estimate: A rough price estimation based on size and quality, using a baseline price of $120/sqft.
    X["Price_Size_Estimate"] = X["Square_Footage"] * X["Neighborhood_Quality"] * 120

    # Land Value Ratio: Ratio of the Lot Size (sqm) to the house's Square Footage.
    # Measures the size of the surrounding land relative to the house footprint.
    X["Land_Value_Ratio"] = np.where(
        X["Square_Footage"] > 0,
        X["Lot_Size_sqm"] / X["Square_Footage"],
        np.nan
    )

    # Age Decay Factor: A non-linear factor to diminish value based on house age: (1 / (1 + House Age)).
    X["Age_Decay_Factor"] = 1 / (1 + X["House_Age"])

    # Modern Ratio: Ratio of modern amenities (Garage + Bathrooms) to the house's Age.
    X["Modern_Ratio"] = np.where(
        X["House_Age"] > 0,
        (X["Garage_Size"] + X["Num_Bathrooms"]) / X["House_Age"],
        np.nan
    )

    # Neighborhood Space Index: Neighborhood Quality multiplied by the natural logarithm of Square Footage (log1p).
    # Log transformation reduces the impact of extreme outliers in large areas.
    X["Neighborhood_Space_Index"] = X["Neighborhood_Quality"] * np.log1p(X["Square_Footage"])

    return X

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Temporary data to detect column types
temp = add_features(X_train)

num_cols = temp.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = temp.select_dtypes(include=["object", "category"]).columns.tolist()

# Define full pipeline
preprocess_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(add_features)),
    ("transform", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]))
])

print("🔥 Preprocessing Pipeline compiled successfully!")


🔥 Preprocessing Pipeline compiled successfully!


In [ ]:
# Apply preprocessing on train
X_train_processed = preprocess_pipeline.fit_transform(X_train)

# Apply on validation
X_val_processed = preprocess_pipeline.transform(X_val)

# Apply on test
X_test_processed = preprocess_pipeline.transform(X_test)


In [ ]:
np.save("X_train_processed.npy", X_train_processed)
np.save("X_val_processed.npy", X_val_processed)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)

In [ ]:
import joblib
joblib.dump(preprocess_pipeline, "preprocessing_pipeline.joblib")

['preprocessing_pipeline.joblib']

In [ ]:
X_test.to_csv("X_test_raw.csv", index=False)
y_test.to_csv("y_test.csv", index=False)